# tiny-tutor: Base vs Fine-tuned Output Comparison

Loads `results/eval_report.json` and renders side-by-side comparisons.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML

In [ ]:
report = json.loads(Path('../results/eval_report.json').read_text())
summary = report['summary']
results = report['results']

print(f"Total evaluated: {summary['n']}")
print(f"Fine-tuned wins: {summary['finetuned_wins']} ({summary['finetuned_win_rate']*100:.1f}%)")
print(f"Base wins:       {summary['base_wins']}")
print(f"Ties:            {summary['ties']}")
print()
print(f"Avg Flesch Reading Ease — fine-tuned: {summary['avg_readability_finetuned']}")
print(f"Avg Flesch Reading Ease — base:       {summary['avg_readability_base']}")

In [ ]:
def render_comparison(result):
    concept = result['concept']
    winner = result['winner']
    base = result['base_explanation']
    ft = result['finetuned_explanation']
    reasoning = result['judgment']['reasoning']
    
    base_color = '#d4edda' if winner == 'base' else '#f8f9fa'
    ft_color = '#d4edda' if winner == 'finetuned' else '#f8f9fa'
    if winner == 'tie':
        base_color = ft_color = '#fff3cd'
    
    html = f"""
    <div style='border:1px solid #ddd; padding:12px; margin-bottom:16px; border-radius:6px'>
      <h4 style='margin:0 0 8px'>{concept} 
        <span style='font-weight:normal;font-size:0.85em;color:#666'>Winner: {winner}</span>
      </h4>
      <div style='display:grid;grid-template-columns:1fr 1fr;gap:12px'>
        <div style='background:{base_color};padding:10px;border-radius:4px'>
          <b>Base model</b><br>{base}
        </div>
        <div style='background:{ft_color};padding:10px;border-radius:4px'>
          <b>Fine-tuned</b><br>{ft}
        </div>
      </div>
      <p style='margin:8px 0 0;font-size:0.85em;color:#555'><i>Judge: {reasoning}</i></p>
    </div>
    """
    return HTML(html)

In [ ]:
# Show first 20 results
for r in results[:20]:
    display(render_comparison(r))

In [ ]:
# Readability distribution
df = pd.DataFrame([
    {
        'concept': r['concept'],
        'flesch_base': r['readability_base']['flesch_reading_ease'],
        'flesch_ft': r['readability_finetuned']['flesch_reading_ease'],
        'words_base': r['readability_base']['word_count'],
        'words_ft': r['readability_finetuned']['word_count'],
        'winner': r['winner'],
    }
    for r in results
])

display(df.describe())
df[['flesch_base', 'flesch_ft']].plot.hist(bins=20, alpha=0.7, title='Flesch Reading Ease Distribution')